#Correlação de Pearson
##Introdução


In [70]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [71]:
caminho = '/content/drive/MyDrive/dados-erva-mate/'

AnaliseSensorialM = pd.read_csv(caminho + 'Análise sensorial.csv', sep=',', decimal='.', encoding='utf-8')
CorM = pd.read_csv(caminho + 'Cor.csv', sep=',', decimal='.', encoding='utf-8')
FenoisIndividuaisM = pd.read_csv(caminho + 'Fenóis individuais.csv', sep=';', decimal='.', encoding='utf-8')
EstatUmidM = pd.read_csv(caminho + 'Estat. Umid e AW.csv', sep=',', decimal='.', encoding='utf-8')

In [72]:
caminho = '/content/drive/MyDrive/dados-erva-mate/'

AnaliseSensorialM = pd.read_csv(
    caminho + 'Análise sensorial.csv',
    sep=',',
    decimal=',', # Changed from '.' to ','
    encoding='utf-8'
)

CorM = pd.read_csv(
    caminho + 'Cor.csv',
    sep=',',
    decimal=',', # Changed from '.' to ','
    encoding='utf-8'
)

EstatUmidM = pd.read_csv(
    caminho + 'Estat. Umid e AW.csv',
    sep=',',
    decimal=',', # Changed from '.' to ','
    encoding='utf-8'
)

# --- Robust Fix for FenoisIndividuaisM header and data types ---
# Read with header=None to treat all rows as data initially
FenoisIndividuaisM_raw = pd.read_csv(
    caminho + 'Fenóis individuais.csv',
    sep=';',
    encoding='utf-8',
    header=None
)

# The actual column names are in the second row (index 1) of the raw data.
# Get names from row 1, strip whitespace, and filter out any NaN values.
new_column_names = [col.strip() for col in FenoisIndividuaisM_raw.iloc[1].tolist() if pd.notna(col)]

# The actual data starts from the third row (index 2).
# Select data for the columns that we have names for (up to the number of determined column names).
FenoisIndividuaisM = FenoisIndividuaisM_raw.iloc[2:, :len(new_column_names)].copy()

# Assign the extracted unique column names.
FenoisIndividuaisM.columns = new_column_names

# Rename 'Trat. Meses' to 'Trat_Meses' for consistency
if 'Trat. Meses' in FenoisIndividuaisM.columns:
    FenoisIndividuaisM = FenoisIndividuaisM.rename(columns={'Trat. Meses': 'Trat_Meses'})

# Convert all columns to numeric, handling comma decimals
for col in FenoisIndividuaisM.columns:
    # Ensure all values are strings before replacement; then replace comma with dot.
    FenoisIndividuaisM[col] = FenoisIndividuaisM[col].astype(str).str.replace(',', '.', regex=False)
    # Convert to numeric, coercing errors to NaN.
    FenoisIndividuaisM[col] = pd.to_numeric(FenoisIndividuaisM[col], errors='coerce')
# ----------------------------------------------------------------

#Correlação: Análise Sensorial


In [73]:
df = AnaliseSensorialM

df_numerico = df.drop(columns='Trat_Meses', errors='ignore')

matriz_correlacao = df_numerico.corr(method='pearson')
print(matriz_correlacao)

                  Green_color  Fresh_green_odor  Sweet_aroma  Green_aroma  \
Green_color          1.000000          0.881140     0.938282     0.902964   
Fresh_green_odor     0.881140          1.000000     0.976002     0.882756   
Sweet_aroma          0.938282          0.976002     1.000000     0.954780   
Green_aroma          0.902964          0.882756     0.954780     1.000000   
Bitter_aroma        -0.975497         -0.915091    -0.953366    -0.908393   
Gold_color          -0.838061         -0.962684    -0.971344    -0.936878   
Straw_odor          -0.865877         -0.822106    -0.919830    -0.979794   
Straw_aroma         -0.944659         -0.946296    -0.944777    -0.828080   
Astringency         -0.911569         -0.968316    -0.966202    -0.889397   

                  Bitter_aroma  Gold_color  Straw_odor  Straw_aroma  \
Green_color          -0.975497   -0.838061   -0.865877    -0.944659   
Fresh_green_odor     -0.915091   -0.962684   -0.822106    -0.946296   
Sweet_aroma     

In [74]:
fig = px.imshow(
    matriz_correlacao,
    text_auto='.2f',          # mostra os valores com 2 casas decimais
    color_continuous_scale='Geyser',
    zmin=-1, zmax=1,
    title='Correlação de Pearson: Análise Sensorial'
)

fig.update_layout(
    width=600,
    height=600,
)

fig.show()

Quanto mais verde, mais amarga.

#Correlação: Cores
## Significado das variáveis na Planilha de Cor

- **Trat_Meses** (Meses de tratamento): tempo de armazenamento, eixo temporal da análise.
- **L** (Luminosidade / L*): claridade da bebida. Varia de 0 (preto) a 100 (branco).
- **a** (Coordenada a*): vermelho vs. verde. Valores positivos indicam maior vermelhidão, negativos maior presença de verde.
- **b** (Coordenada b*): amarelo vs. azul. Valores positivos indicam amarelamento, negativos tonalidade azulada.
- **C** (Chroma / C*): saturação da cor. Quanto maior, mais viva e intensa a coloração.
- **D** (De-greening / D*): perda de coloração verde, associada à degradação da clorofila ao longo do armazenamento.
- **YI** (Yellowing Index): índice de amarelamento. Tende a aumentar conforme a bebida envelhece.
- **DeltaE** (ΔE): diferença total de cor em relação a uma referência. Quanto maior, mais a cor se afastou do estado original.

> Fonte: CÂMARA, O. P. et al. NIX quality control colorimeter can evaluate color of yerba mate. *Food Science and Technology*, Campinas, v. 45, e00460, 2025.

In [75]:
df = CorM

df_numerico = df.drop(columns='Trat_Meses', errors='ignore')

matriz_correlacao = df_numerico.corr(method='pearson')
print(matriz_correlacao)

               L         a         b         C         D        YI    DeltaE
L       1.000000  0.900762  0.156780  0.156478  0.909759 -0.463683  0.973211
a       0.900762  1.000000 -0.069290 -0.074082  0.996638 -0.613267  0.826048
b       0.156780 -0.069290  1.000000  0.999717 -0.013602  0.801976  0.379188
C       0.156478 -0.074082  0.999717  1.000000 -0.018946  0.802371  0.379155
D       0.909759  0.996638 -0.013602 -0.018946  1.000000 -0.568332  0.846775
YI     -0.463683 -0.613267  0.801976  0.802371 -0.568332  1.000000 -0.247902
DeltaE  0.973211  0.826048  0.379188  0.379155  0.846775 -0.247902  1.000000


In [76]:
fig = px.imshow(
    matriz_correlacao,
    text_auto='.2f',          # mostra os valores com 2 casas decimais
    color_continuous_scale='Geyser',
    zmin=-1, zmax=1,
    title='Correlação de Pearson: Cor'
)

fig.update_layout(
    width=600,
    height=600,
)

fig.show()

#Correlação: Fenóis Individuais


In [77]:
df = FenoisIndividuaisM

df_numerico = df.drop(columns='Trat_Meses', errors='ignore')

matriz_correlacao = df_numerico.corr(method='pearson')
print(matriz_correlacao)

                  Caffeine  Theobromine  Catechin  Chlorogenic Acid  \
Caffeine          1.000000    -0.347896  0.853788         -0.525070   
Theobromine      -0.347896     1.000000  0.089095          0.826838   
Catechin          0.853788     0.089095  1.000000         -0.164776   
Chlorogenic Acid -0.525070     0.826838 -0.164776          1.000000   
Caffeic Acid      0.456855    -0.736990  0.062995         -0.607628   

                  Caffeic Acid  
Caffeine              0.456855  
Theobromine          -0.736990  
Catechin              0.062995  
Chlorogenic Acid     -0.607628  
Caffeic Acid          1.000000  


In [78]:
fig = px.imshow(
    matriz_correlacao,
    text_auto='.2f',          # mostra os valores com 2 casas decimais
    color_continuous_scale='Geyser',
    zmin=-1, zmax=1,
    title='Correlação de Pearson: Fenóis Individuais'
)

fig.update_layout(
    width=600,
    height=600,
)

fig.show()

#Correlação: Total

In [79]:
df = pd.concat([AnaliseSensorialM, CorM, FenoisIndividuaisM, EstatUmidM], axis=1)

df_numerico = df.drop(columns='Trat_Meses', errors='ignore')

matriz_correlacao = df_numerico.corr(method='pearson')
print(matriz_correlacao)

                  Green_color  Fresh_green_odor  Sweet_aroma  Green_aroma  \
Green_color          1.000000          0.881140     0.938282     0.902964   
Fresh_green_odor     0.881140          1.000000     0.976002     0.882756   
Sweet_aroma          0.938282          0.976002     1.000000     0.954780   
Green_aroma          0.902964          0.882756     0.954780     1.000000   
Bitter_aroma        -0.975497         -0.915091    -0.953366    -0.908393   
Gold_color          -0.838061         -0.962684    -0.971344    -0.936878   
Straw_odor          -0.865877         -0.822106    -0.919830    -0.979794   
Straw_aroma         -0.944659         -0.946296    -0.944777    -0.828080   
Astringency         -0.911569         -0.968316    -0.966202    -0.889397   
L                   -0.875311         -0.620691    -0.763408    -0.817696   
a                   -0.984588         -0.848101    -0.929048    -0.945249   
b                    0.214848          0.382775     0.244820    -0.016531   

In [81]:
fig = px.imshow(
    matriz_correlacao,
    text_auto='.2f',          # mostra os valores com 2 casas decimais
    color_continuous_scale='Geyser',
    zmin=-1, zmax=1,
    title='Correlação de Pearson: Total'
)

fig.update_layout(
    width=600,
    height=600,
    font=dict(size=10)
)

fig.show()